# NB16 — Constrained Cooling OE Calibration

**Goal**: Calibrate `outlet_effectiveness` for cooling mode using `cooling_training_data.csv.gz`,
with HLC and τ locked from heating calibration (building/slab physics are mode-invariant).

**Key findings from NB10/NB15**:
- HLC = 0.1206 kW/K is a building envelope property → same in both modes (scipy confirms)
- τ = 4.84h is a slab time constant → same physical mass in both modes
- OE = 0.95 in cooling state file is WRONG (config default, never properly calibrated)
- HP saturates at 18°C min outlet → OE unidentifiable in saturated regime
- Expected OE_cooling ≈ 0.15–0.25 (passive convection vs forced 0.83 in heating)

**Method**: scipy.optimize.minimize_scalar on equilibrium RMSE, constrained to OE only.

In [ ]:
import numpy as np
import pandas as pd
from scipy import optimize
import matplotlib.pyplot as plt

# Locked heating-calibrated parameters (building/slab physics, mode-invariant)
HLC_LOCKED = 0.1206       # kW/K — heating calibration, confirmed by NB10 scipy
TAU_LOCKED = 4.836        # hours — slab time constant
DT_H = 5.0 / 60.0        # 5-minute timestep in hours
EXP_FACTOR = 1 - np.exp(-DT_H / TAU_LOCKED)

# Config
MAX_AT = 45.0             # filter AT sensor glitches (raw data has 839°C!)
MIN_DRIVE_K = 0.5         # min (T_indoor - T_outlet) for identifiability

## Phase A: Load and clean cooling training data

In [ ]:
df_raw = pd.read_csv('../../Logs_and_models/cooling_training_data.csv.gz')
print(f'Raw: {len(df_raw)} rows')

# Quality filter
mask = (
    (df_raw['AT'] > -15) & (df_raw['AT'] < MAX_AT) &
    (df_raw['VLT'] > 10) & (df_raw['VLT'] < 40) &
    (df_raw['RLT'] > 10) & (df_raw['RLT'] < 40) &
    (df_raw['indoor_temp'] > 15) & (df_raw['indoor_temp'] < 35)
)
df = df_raw[mask].copy().reset_index(drop=True)
print(f'After quality filter: {len(df)} rows (removed {len(df_raw)-len(df)} bad)')

# Segment by operating mode
df_cool = df[df['thermal_power_kw'] < -0.1].copy()
df_idle = df[(df['thermal_power_kw'] >= -0.1) & (df['thermal_power_kw'] <= 0.1)].copy()
df_heat = df[df['thermal_power_kw'] > 0.1].copy()
print(f'Active cooling: {len(df_cool)}, Idle: {len(df_idle)}, Heating: {len(df_heat)}')

# Derived columns
df_cool['drive'] = df_cool['indoor_temp'] - df_cool['VLT']
df_cool['outdoor_load'] = df_cool['AT'] - df_cool['indoor_temp']

## Phase B: Constrained OE calibration (temperature-based)

In [ ]:
def compute_teq_residual(oe, vlt, at, indoor, a0=0.0):
    """T_eq = (OE*VLT + HLC*AT)/(OE+HLC) + a0; return indoor - T_eq."""
    denom = oe + HLC_LOCKED
    t_eq = (oe * vlt + HLC_LOCKED * at) / denom + a0
    return indoor - t_eq

def rmse_oe(params, vlt, at, indoor, with_a0=False):
    """RMSE objective for OE (+ optional a0) calibration."""
    if with_a0:
        oe, a0 = params
    else:
        oe, a0 = params[0], 0.0
    res = compute_teq_residual(oe, vlt, at, indoor, a0)
    return np.sqrt(np.mean(res**2))

# Calibration across multiple dataset slices
datasets = {
    'All active cooling': df_cool,
    'Non-saturated (VLT>19, drive>1K)': df_cool[(df_cool['VLT'] > 19) & (df_cool['drive'] > MIN_DRIVE_K)],
    'Non-saturated (drive>2K)': df_cool[df_cool['drive'] > 2.0],
    'Night cooling (PV<50W)': df_cool[df_cool['PV_Generate'] < 50],
    'Day cooling (PV>500W)': df_cool[df_cool['PV_Generate'] > 500],
    'Moderate outdoor (AT<28)': df_cool[df_cool['AT'] < 28],
    'Hot outdoor (AT>=28)': df_cool[df_cool['AT'] >= 28],
}

results = []
print(f"{'Dataset':>40s}  {'N':>6s}  {'OE':>8s}  {'RMSE':>7s}  {'Bias':>7s}  {'OE+a0':>8s}  {'a0':>7s}  {'RMSE2':>7s}")
print('-' * 100)

for name, subset in datasets.items():
    if len(subset) < 50:
        print(f'{name:>40s}  {len(subset):6d}  (too few)')
        continue
    vlt, at, indoor = subset['VLT'].values, subset['AT'].values, subset['indoor_temp'].values
    
    # OE only
    r1 = optimize.minimize_scalar(lambda oe: rmse_oe([oe], vlt, at, indoor), bounds=(0.01, 2.0), method='bounded')
    bias1 = np.mean(compute_teq_residual(r1.x, vlt, at, indoor))
    
    # OE + a0
    r2 = optimize.minimize(rmse_oe, [r1.x, 0.0], args=(vlt, at, indoor, True),
                           bounds=[(0.01, 2.0), (-5, 5)], method='L-BFGS-B')
    
    print(f'{name:>40s}  {len(subset):6d}  {r1.x:8.4f}  {r1.fun:7.4f}  {bias1:+7.4f}  '
          f'{r2.x[0]:8.4f}  {r2.x[1]:+7.4f}  {r2.fun:7.4f}')
    results.append(dict(name=name, n=len(subset), oe=r1.x, rmse=r1.fun, bias=bias1,
                        oe_a0=r2.x[0], a0=r2.x[1], rmse_a0=r2.fun))

## Phase C: OE sensitivity sweep

In [ ]:
# Sweep OE over non-saturated cooling data
cal = df_cool[df_cool['drive'] > MIN_DRIVE_K].copy()
vlt, at, indoor = cal['VLT'].values, cal['AT'].values, cal['indoor_temp'].values

oe_sweep = np.arange(0.05, 1.05, 0.02)
sweep = []
for oe in oe_sweep:
    res = compute_teq_residual(oe, vlt, at, indoor)
    sweep.append((oe, np.sqrt(np.mean(res**2)), np.mean(res), np.mean(np.abs(res))))
sweep = pd.DataFrame(sweep, columns=['OE', 'RMSE', 'Bias', 'MAE'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(sweep['OE'], sweep['RMSE'], 'b-', lw=2, label='RMSE')
ax1.plot(sweep['OE'], sweep['MAE'], 'g--', lw=2, label='MAE')
ax1.axvline(sweep.loc[sweep['RMSE'].idxmin(), 'OE'], color='r', ls=':', label=f'Best OE={sweep.loc[sweep["RMSE"].idxmin(), "OE"]:.3f}')
ax1.axvline(0.953, color='orange', ls='--', alpha=0.7, label='Production (0.953)')
ax1.axvline(0.826, color='purple', ls='--', alpha=0.7, label='Heating (0.826)')
ax1.set_xlabel('OE'); ax1.set_ylabel('Error (°C)'); ax1.set_title('OE Sensitivity (non-saturated cooling)')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(sweep['OE'], sweep['Bias'], 'r-', lw=2)
ax2.axhline(0, color='k', ls='-', alpha=0.3)
ax2.axvline(sweep.loc[sweep['RMSE'].idxmin(), 'OE'], color='r', ls=':')
ax2.set_xlabel('OE'); ax2.set_ylabel('Bias (°C)'); ax2.set_title('Bias vs OE')
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nBest OE (min RMSE): {sweep.loc[sweep['RMSE'].idxmin(), 'OE']:.3f} → RMSE={sweep['RMSE'].min():.4f}°C")

## Phase D: Production vs calibrated comparison

In [ ]:
# Best OE from Phase B results
best = [r for r in results if 'drive>1K' in r['name'] or 'Non-saturated' in r['name']]
oe_best = best[0]['oe'] if best else results[0]['oe']
oe_best_a0, a0_best = (best[0]['oe_a0'], best[0]['a0']) if best else (oe_best, 0.0)

configs = [
    ('Production (OE=0.953)', 0.953, 0.0),
    ('NB10 calibrated (OE=0.190)', 0.190, 0.0),
    (f'NB16 best OE={oe_best:.3f}', oe_best, 0.0),
    (f'NB16 OE={oe_best_a0:.3f}+a0={a0_best:+.3f}', oe_best_a0, a0_best),
    ('Heating (OE=0.826)', 0.826, 0.0),
]

print(f"{'Config':>45s}  {'RMSE':>7s}  {'Bias':>7s}  {'MAE':>7s}  {'P95':>7s}")
print('-' * 80)
for name, oe, a0 in configs:
    res = compute_teq_residual(oe, vlt, at, indoor, a0)
    print(f'{name:>45s}  {np.sqrt(np.mean(res**2)):7.4f}  {np.mean(res):+7.4f}  '
          f'{np.mean(np.abs(res)):7.4f}  {np.percentile(np.abs(res), 95):7.4f}')

## Phase E: Dual-HLC analysis

In [ ]:
# Dual-HLC: separate HLC for HP-ON vs HP-OFF
all_clean = df[df['AT'] < MAX_AT].copy()
all_clean['hp_on'] = all_clean['thermal_power_kw'].abs() > 0.1

va, aa, ia, ha = all_clean['VLT'].values, all_clean['AT'].values, all_clean['indoor_temp'].values, all_clean['hp_on'].values

def single_hlc_rmse(params, v, a, i):
    hlc, oe = params
    return np.sqrt(np.mean((i - (oe*v + hlc*a)/(oe+hlc))**2))

def dual_hlc_rmse(params, v, a, i, h):
    hlc_on, hlc_off, oe = params
    hlc = np.where(h, hlc_on, hlc_off)
    return np.sqrt(np.mean((i - (oe*v + hlc*a)/(oe+hlc))**2))

r_s = optimize.minimize(single_hlc_rmse, [0.12, 0.5], args=(va, aa, ia),
                        bounds=[(0.005, 0.5), (0.01, 2.0)], method='L-BFGS-B')
r_d = optimize.minimize(dual_hlc_rmse, [0.15, 0.02, 0.5], args=(va, aa, ia, ha),
                        bounds=[(0.02, 0.5), (0.005, 0.3), (0.01, 2.0)], method='L-BFGS-B')

print(f'Single-HLC: HLC={r_s.x[0]:.4f}, OE={r_s.x[1]:.4f}, RMSE={r_s.fun:.4f}')
print(f'Dual-HLC:   HLC_on={r_d.x[0]:.4f}, HLC_off={r_d.x[1]:.4f}, OE={r_d.x[2]:.4f}, RMSE={r_d.fun:.4f}')
print(f'Improvement: {(1 - r_d.fun/r_s.fun)*100:.1f}%')

## Summary

| Parameter | Production | Calibrated | Source |
|-----------|-----------|------------|--------|
| HLC | 0.1245 (cooling state) | 0.1206 | Locked from heating (building property) |
| τ | 4.39h (cooling state) | 4.84h | Locked from heating (slab property) |
| OE | 0.953 (WRONG) | ~0.20 | NB16 scipy constrained calibration |

**Root cause**: OE=0.95 was never calibrated — it's the config default (0.90) drifted upward
by online learning that couldn't converge because HLC/OE are confounded when the HP
saturates at its 18°C minimum outlet temperature.

**Fix applied**: `physics_calibration_cooling.py` now locks HLC and τ from heating state
and calibrates only OE via scipy RMSE minimisation on non-saturated data.